# TRELLIS.2 — RunPod GPU Setup & Inference

Tested on **A100 / H100 (24 GB+ VRAM)** with CUDA 12.4.  
Run cells top-to-bottom on a fresh RunPod PyTorch pod.

## 1. Verify GPU

In [ ]:
!nvidia-smi

In [ ]:
!nvcc --version

## 2. Clone Repo (skip if already present)

In [ ]:
import os

REPO_DIR = "/workspace/TRELLIS.2"

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/microsoft/TRELLIS.git {REPO_DIR}
else:
    print(f"Repo already present at {REPO_DIR}")

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 3. Install Dependencies

This mirrors `setup.sh`. Split into steps so you can re-run individual parts if one fails.

### 3a. PyTorch (CUDA 12.4)

In [ ]:
!pip install torch==2.6.0 torchvision==0.21.0 --index-url https://download.pytorch.org/whl/cu124 -q

### 3b. Basic Python packages

In [ ]:
!pip install imageio imageio-ffmpeg tqdm easydict opencv-python-headless ninja \
             trimesh transformers gradio==6.0.1 tensorboard pandas lpips zstandard \
             kornia timm -q
!pip install git+https://github.com/EasternJournalist/utils3d.git@9a4eb15e4021b67b12c460c7057d642626897ec8 -q

In [ ]:
# Install system libs needed by pillow-simd (package names vary by distro/image)
!apt-get update -qq && apt-get install -y libjpeg-turbo8-dev zlib1g-dev -qq 2>/dev/null || \
 apt-get install -y libjpeg-dev zlib1g-dev -qq 2>/dev/null || \
 echo "apt install skipped (packages may already be present)"

# pillow-simd is a drop-in faster Pillow; fall back to stock Pillow if it fails
!pip install pillow-simd -q 2>/dev/null || pip install Pillow -q
!python -c "from PIL import Image; print('Pillow OK:', Image.__version__)"

### 3c. Flash-Attention

In [ ]:
!pip install flash-attn==2.7.3 -q

### 3d. CUDA extensions (nvdiffrast, nvdiffrec, CuMesh, FlexGEMM, O-Voxel)

These compile from source — expect a few minutes each.

In [ ]:
%%bash
set -e
mkdir -p /tmp/extensions

echo "==> nvdiffrast"
[ -d /tmp/extensions/nvdiffrast ] || git clone -b v0.4.0 https://github.com/NVlabs/nvdiffrast.git /tmp/extensions/nvdiffrast
pip install /tmp/extensions/nvdiffrast --no-build-isolation -q

echo "==> nvdiffrec"
[ -d /tmp/extensions/nvdiffrec ] || git clone -b renderutils https://github.com/JeffreyXiang/nvdiffrec.git /tmp/extensions/nvdiffrec
pip install /tmp/extensions/nvdiffrec --no-build-isolation -q

echo "==> CuMesh"
[ -d /tmp/extensions/CuMesh ] || git clone https://github.com/JeffreyXiang/CuMesh.git /tmp/extensions/CuMesh --recursive
pip install /tmp/extensions/CuMesh --no-build-isolation -q

echo "==> FlexGEMM"
[ -d /tmp/extensions/FlexGEMM ] || git clone https://github.com/JeffreyXiang/FlexGEMM.git /tmp/extensions/FlexGEMM --recursive
pip install /tmp/extensions/FlexGEMM --no-build-isolation -q

echo "==> O-Voxel"
cp -r /workspace/TRELLIS.2/o-voxel /tmp/extensions/o-voxel 2>/dev/null || true
pip install /tmp/extensions/o-voxel --no-build-isolation -q

echo "All extensions installed."

## 4. Verify Imports

In [ ]:
import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
import sys
sys.path.insert(0, '/workspace/TRELLIS.2')

import cv2
import imageio
from PIL import Image
import o_voxel
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap

print("All imports OK")

## 5. Load Pipeline

Downloads ~4B-parameter model weights from HuggingFace on first run (~15 GB).

In [ ]:
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()
print("Pipeline ready.")

## 6. Load Environment Map (for PBR rendering)

In [ ]:
envmap = EnvMap(
    torch.tensor(
        cv2.cvtColor(
            cv2.imread('assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED),
            cv2.COLOR_BGR2RGB
        ),
        dtype=torch.float32,
        device='cuda'
    )
)
print("Environment map loaded.")

## 7. Pick Input Images

Any PNG / WebP with a clean background works.  
Change `image_paths` to your own files or upload them to `/workspace/`.

In [ ]:
import glob
from IPython.display import display

# Use a handful of the bundled example images
all_examples = sorted(glob.glob('assets/example_image/*.webp'))[:4]
all_examples += ['assets/example_image/T.png']  # also include the PNG

print("Images to process:")
for p in all_examples:
    img = Image.open(p).convert('RGB')
    img.thumbnail((128, 128))
    display(img)
    print(p)

## 8. Run Inference

Default pipeline type is `'1024_cascade'` (best quality).  
Use `'512'` for a quick test (~30 s vs ~3 min on A100).

In [ ]:
import time

PIPELINE_TYPE = '1024_cascade'  # '512' for fast test
OUTPUT_DIR = '/workspace/trellis2_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for img_path in all_examples:
    name = os.path.splitext(os.path.basename(img_path))[0]
    print(f"\n{'='*60}")
    print(f"Processing: {img_path}")

    image = Image.open(img_path)

    t0 = time.time()
    mesh = pipeline.run(image, pipeline_type=PIPELINE_TYPE)[0]
    mesh.simplify(16_777_216)  # nvdiffrast vertex limit
    print(f"Inference: {time.time()-t0:.1f}s | "
          f"vertices={mesh.vertices.shape[0]:,} faces={mesh.faces.shape[0]:,}")

    # --- Render turntable video ---
    t1 = time.time()
    frames = render_utils.make_pbr_vis_frames(
        render_utils.render_video(mesh, envmap=envmap)
    )
    video_path = os.path.join(OUTPUT_DIR, f'{name}.mp4')
    imageio.mimsave(video_path, frames, fps=15)
    print(f"Video saved: {video_path}  ({time.time()-t1:.1f}s)")

    # --- Export GLB ---
    t2 = time.time()
    glb = o_voxel.postprocess.to_glb(
        vertices          = mesh.vertices,
        faces             = mesh.faces,
        attr_volume       = mesh.attrs,
        coords            = mesh.coords,
        attr_layout       = mesh.layout,
        voxel_size        = mesh.voxel_size,
        aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target = 1_000_000,
        texture_size      = 4096,
        remesh            = True,
        remesh_band       = 1,
        remesh_project    = 0,
        use_tqdm          = True,
    )
    glb_path = os.path.join(OUTPUT_DIR, f'{name}.glb')
    glb.export(glb_path, extension_webp=True)
    print(f"GLB saved:   {glb_path}  ({time.time()-t2:.1f}s)")

    # --- Export USDZ ---
    t3 = time.time()
    usdz_path = os.path.join(OUTPUT_DIR, f'{name}.usdz')
    o_voxel.postprocess.to_usdz(
        vertices          = mesh.vertices,
        faces             = mesh.faces,
        attr_volume       = mesh.attrs,
        coords            = mesh.coords,
        attr_layout       = mesh.layout,
        voxel_size        = mesh.voxel_size,
        aabb              = [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target = 1_000_000,
        texture_size      = 4096,
        remesh            = True,
        remesh_band       = 1,
        remesh_project    = 0,
        file_path         = usdz_path,
        use_tqdm          = True,
    )
    print(f"USDZ saved:  {usdz_path}  ({time.time()-t3:.1f}s)")

    torch.cuda.empty_cache()

print("\nAll done.")

## 9. Preview Outputs

In [ ]:
# List generated files
!ls -lh {OUTPUT_DIR}

In [ ]:
# Preview the first video inline
from IPython.display import Video
import glob

videos = sorted(glob.glob(f'{OUTPUT_DIR}/*.mp4'))
if videos:
    display(Video(videos[0], embed=True, width=512))

## 10. Download Outputs

Zip everything up for easy download from the RunPod file manager.

In [ ]:
!zip -r /workspace/trellis2_outputs.zip {OUTPUT_DIR}
print("Ready: /workspace/trellis2_outputs.zip")

---
## Optional: Single-Image Quick Test

Drop your own image at `/workspace/my_image.png` and run this cell.

In [ ]:
MY_IMAGE = '/workspace/my_image.png'  # <-- change this

if os.path.exists(MY_IMAGE):
    image = Image.open(MY_IMAGE)
    mesh = pipeline.run(image, pipeline_type='512')[0]  # '512' for speed
    mesh.simplify(16_777_216)

    glb = o_voxel.postprocess.to_glb(
        vertices=mesh.vertices, faces=mesh.faces,
        attr_volume=mesh.attrs, coords=mesh.coords,
        attr_layout=mesh.layout, voxel_size=mesh.voxel_size,
        aabb=[[-0.5,-0.5,-0.5],[0.5,0.5,0.5]],
        decimation_target=500_000, texture_size=2048,
        remesh=True, remesh_band=1, remesh_project=0,
    )
    out = '/workspace/my_output.glb'
    glb.export(out, extension_webp=True)
    print(f"Saved: {out}")

    torch.cuda.empty_cache()
else:
    print(f"File not found: {MY_IMAGE}")